In [2]:
import math

In [ ]:
# ------------------------------------------------------------------
# VARIABLES GLOBALES — Caso de Estudio 1: Flexocompresión (NTC-2023)
# Unidades: kg, cm
# ------------------------------------------------------------------

# Perfil IR 305x74.5
d    = 30.5      # profundidad total del perfil, cm
b_p  = 16.5      # ancho de ala, cm
F_y  = 3515.0    # resistencia a fluencia del acero, kg/cm² (A992)

# Placa base
N = 50.0         # dimensión de la placa en la dirección del momento, cm
B = 50.0         # dimensión de la placa perpendicular al momento, cm

# Dado (pedestal) de concreto
dado_x = 60.0    # cm
dado_y = 60.0    # cm
f_c    = 250.0   # resistencia especificada del concreto, kg/cm²

# Cargas de diseño
P_u = 12000.0    # carga axial última (compresión), kg
M_u = 600000.0   # momento último, kg·cm  (6 ton·m)
V_u = 4000.0     # cortante último, kg

# Geometría de anclas
f               = 20.0  # distancia del eje de la placa al centro de anclas en tensión, cm
num_anclas_tension = 2  # número de anclas sujetas a tensión

# Factores de resistencia (NTC-2023)
F_R_aplastamiento = 0.65   # Sección 13.3.1
F_R_flexion       = 0.90   # Ec. 13.2.1


## Bloque 1 — Áreas, Esfuerzo de Aplastamiento y Voladizos Críticos

### Fundamento teórico (NTC-2023, Cap. 13)

La placa base transmite las cargas de la columna al pedestal de concreto mediante **esfuerzo de aplastamiento**. La norma limita este esfuerzo considerando el aumento de confinamiento cuando el área del pedestal $A_2$ supera el área de la placa $A_1$.

#### Esfuerzo de aplastamiento (Sección 13.3.1, Ec. 13.3.1.b)

$$f_{pu} = 0.85\, f'_c \sqrt{\dfrac{A_2}{A_1}} \quad \leq 1.7\, f'_c$$

El esfuerzo de diseño se obtiene aplicando el factor de resistencia $F_R$:

$$f_{pd} = F_R\, f_{pu}$$

#### Voladizos críticos (Sección 13.1.1.4)

Los voladizos miden la longitud de placa que sobresale de la huella del perfil sobre la placa. El voladizo crítico $l$ rige el cálculo del momento flector y del espesor mínimo:

$$m = \dfrac{N - b_p}{2}, \qquad n = \dfrac{B - d}{2}, \qquad l = \max(m,\, n)$$

In [4]:
# ------------------------------------------------------------------
# BLOQUE 1 — Áreas, aplastamiento y voladizos críticos
# ------------------------------------------------------------------

# 1) Áreas (Ec. 13.3.1.b)
A1 = N * B             # cm², área de la placa base
A2 = dado_x * dado_y   # cm², área del dado de concreto

# 2) Esfuerzo de aplastamiento (Ec. 13.3.1.b, límite: 1.7 f'c)
f_pu_raw   = 0.85 * f_c * math.sqrt(A2 / A1)
f_pu_limit = 1.7 * f_c
f_pu       = min(f_pu_raw, f_pu_limit)

# 3) Esfuerzo de diseño por aplastamiento (Sección 13.3.1)
f_pd = F_R_aplastamiento * f_pu

# 4) Voladizos críticos (Sección 13.1.1.4)
m = (N - b_p) / 2.0
n = (B - d)   / 2.0
l = max(m, n)

## Bloque 2 — Análisis de Excentricidad

### Fundamento teórico (NTC-2023, Sección 13.1.4.1)

Para determinar el régimen de diseño (pequeña o gran excentricidad) se compara la **excentricidad real** de la carga con una **excentricidad crítica** que delimita ambos regímenes.

#### Excentricidad real

$$e = \dfrac{M_u}{P_u}$$

#### Excentricidad crítica

La excentricidad crítica representa el límite a partir del cual la resultante de compresiones sale del núcleo central de la placa y se requieren anclas en tensión:

$$e_{crit} = \dfrac{N}{2} - \dfrac{P_u}{2\, B\, f_{pd}}$$

Si $e > e_{crit}$ → **gran excentricidad (Caso 1)**: parte de la placa levanta y las anclas deben resistir tensión.

In [5]:
# ------------------------------------------------------------------
# BLOQUE 2 — Análisis de excentricidad (NTC 13.1.4.1)
# ------------------------------------------------------------------

# 1) Excentricidad real
e = M_u / P_u  # cm

# 2) Excentricidad crítica
den = 2.0 * B * f_pd
if den == 0:
    raise ZeroDivisionError("Denominador en e_crit es cero (verificar f_pd y B)")
e_crit = N / 2.0 - P_u / den  # cm

## Bloque 3 — Equilibrio de Fuerzas y Tensión en Anclas

### Fundamento teórico (NTC-2023, Sección 13.1.4.2)

En el caso de gran excentricidad, el bloque de compresiones de longitud $Y$ se ubica en la zona comprimida de la placa. La posición de $Y$ se determina por equilibrio de fuerzas verticales y de momentos respecto al centroide de las anclas en tensión.

#### Longitud del bloque de compresión (Ec. 13.1.4.2.f)

$$Y = \left(f + \dfrac{N}{2}\right) - \sqrt{\left(f + \dfrac{N}{2}\right)^2 - \dfrac{2\, P_u\,(e + f)}{B\, f_{pd}}}$$

donde $f$ es la distancia del eje neutro de la placa al centro del grupo de anclas en tensión.

#### Tensión total en anclas (Ec. 13.1.4.2.h)

Por equilibrio de fuerzas verticales, la tensión total que deben resistir las anclas es:

$$T_{ua} = B\, f_{pd}\, Y - P_u$$

In [6]:
# ------------------------------------------------------------------
# BLOQUE 3 — Equilibrio de fuerzas y tensión en anclas (NTC 13.1.4.2)
# ------------------------------------------------------------------

# Ec. 13.1.4.2.f: longitud del bloque de compresión Y
a = f + N / 2.0
term_inside_sqrt = a**2 - (2.0 * P_u * (e + f)) / (B * f_pd)

if term_inside_sqrt < 0:
    if term_inside_sqrt > -1e-9:   # tolerancia numérica
        term_inside_sqrt = 0.0
    else:
        raise ValueError(f"Discriminante negativo al calcular Y: {term_inside_sqrt:.6f}")

Y = a - math.sqrt(term_inside_sqrt)  # cm

# Ec. 13.1.4.2.h: tensión total requerida en anclas
T_ua = B * f_pd * Y - P_u  # kg

# Demanda por ancla individual
if num_anclas_tension <= 0:
    raise ValueError("El número de anclas en tensión debe ser >= 1")
T_por_ancla = T_ua / num_anclas_tension  # kg

## Bloque 4 — Momento en la Placa y Espesor Mínimo

### Fundamento teórico (NTC-2023, Sección 13.2)

El momento flector en la sección crítica de la placa depende de si el bloque de compresión $Y$ queda dentro o fuera del voladizo crítico $l$.

#### Momento en la placa por unidad de ancho (Ecs. 13.1.4.2.d y 13.1.4.2.e)

La Ec. 13.2.1 trabaja con **momento por unidad de ancho** [kg·cm/cm]. Si $Y \leq l$:

$$M_{u,placa} = f_{pd}\, Y \left(a - \dfrac{Y}{2}\right)$$

Si $Y > l$, el bloque de compresión supera el voladizo crítico y se sustituye $Y$ por $l$:

$$M_{u,placa} = f_{pd}\, l \left(a - \dfrac{l}{2}\right)$$

#### Espesor mínimo requerido (Ec. 13.2.1)

Igualando el momento resistente de la sección transversal de la placa al momento actuante por unidad de ancho:

$$t_{req} = \sqrt{\dfrac{4\, M_{u,placa}}{F_R\, F_y}}$$


In [ ]:
# ------------------------------------------------------------------
# BLOQUE 4 — Momento en la placa y espesor mínimo (NTC 13.2)
# ------------------------------------------------------------------

# Momento por unidad de ancho según Ecs. 13.1.4.2.d y 13.1.4.2.e
# (Ec. 13.2.1 trabaja con [kg·cm/cm], NO con el momento total)
if Y <= l:
    M_u_placa = f_pd * Y * (a - Y / 2.0)  # kg·cm/cm
    caso_momento = "Y ≤ l → se usa Y (Ec. 13.1.4.2.d)"
else:
    M_u_placa = f_pd * l * (a - l / 2.0)  # kg·cm/cm
    caso_momento = "Y > l → se usa l (Ec. 13.1.4.2.e)"

# Espesor mínimo requerido (Ec. 13.2.1)
if F_R_flexion * F_y <= 0:
    raise ValueError("F_R_flexion * F_y no puede ser cero o negativo")
t_req = math.sqrt((4.0 * M_u_placa) / (F_R_flexion * F_y))  # cm


In [ ]:
# ==================================================================
# RESUMEN COMPLETO DE RESULTADOS
# Caso de Estudio 1: Flexocompresión Extrema — NTC-2023, Cap. 13
# Unidades: kg, cm
# ==================================================================

sep = "=" * 65
print(sep)
print("CASO DE ESTUDIO 1 — FLEXOCOMPRESIÓN EXTREMA (NTC-2023, Cap. 13)")
print(sep)

# --- Datos de entrada ---
print("\n--- DATOS DE ENTRADA ---")
print(f"  Perfil IR 305x74.5 : d = {d} cm,  b_p = {b_p} cm")
print(f"  Acero A992         : F_y = {F_y} kg/cm²")
print(f"  Placa base         : N = {N} cm,  B = {B} cm")
print(f"  Dado de concreto   : {dado_x} × {dado_y} cm")
print(f"  Concreto           : f'c = {f_c} kg/cm²")
print(f"  Cargas             : P_u = {P_u:.0f} kg,  M_u = {M_u:.0f} kg·cm,  V_u = {V_u:.0f} kg")
print(f"  Anclas             : f = {f} cm,  n_anclas = {num_anclas_tension}")
print(f"  F_R aplastamiento  : {F_R_aplastamiento},   F_R flexión: {F_R_flexion}")

# --- Bloque 1 ---
print("\n--- BLOQUE 1: Áreas, aplastamiento y voladizos (Sección 13.3.1 / 13.1.1.4) ---")
print(f"  A1 = N × B = {A1:.2f} cm²")
print(f"  A2 = {dado_x} × {dado_y} = {A2:.2f} cm²")
print(f"  f_pu (raw) = 0.85 × f'c × √(A2/A1) = {f_pu_raw:.6f} kg/cm²")
print(f"  Límite 1.7 f'c     = {f_pu_limit:.6f} kg/cm²")
if f_pu_raw > f_pu_limit:
    print(f"  ► Se aplica límite: f_pu = {f_pu:.6f} kg/cm²")
else:
    print(f"  f_pu (usado)       = {f_pu:.6f} kg/cm²")
print(f"  f_pd = F_R × f_pu  = {F_R_aplastamiento} × {f_pu:.6f} = {f_pd:.6f} kg/cm²")
print(f"  m = (N - b_p)/2    = {m:.4f} cm")
print(f"  n = (B - d)/2      = {n:.4f} cm")
print(f"  l = max(m, n)      = {l:.4f} cm")

# --- Bloque 2 ---
print("\n--- BLOQUE 2: Excentricidad (Sección 13.1.4.1) ---")
print(f"  e      = M_u / P_u                        = {e:.4f} cm")
print(f"  e_crit = N/2 - P_u/(2·B·f_pd)             = {e_crit:.4f} cm")
if e > e_crit:
    print("  ► e > e_crit → Gran excentricidad (Caso 1): se requieren anclas en tensión.")
else:
    print("  ► e ≤ e_crit → Pequeña excentricidad: el procedimiento de Caso 1 no aplica.")

# --- Bloque 3 ---
print("\n--- BLOQUE 3: Bloque de compresión y tensión en anclas (Sección 13.1.4.2) ---")
print(f"  a = f + N/2                               = {a:.4f} cm")
print(f"  Discriminante                             = {term_inside_sqrt:.6f}")
print(f"  Y = a - √(discriminante)                  = {Y:.6f} cm")
print(f"  T_ua = B·f_pd·Y - P_u                     = {T_ua:.3f} kg")
print(f"  T_por_ancla (/{num_anclas_tension} anclas) = {T_por_ancla:.3f} kg")
if T_ua <= 0:
    print("  ► ATENCIÓN: T_ua ≤ 0 → no se requieren anclas en tensión.")

# --- Bloque 4 ---
print("\n--- BLOQUE 4: Momento en placa y espesor mínimo (Sección 13.2) ---")
print(f"  {caso_momento}")
print(f"  M_u_placa (por unidad de ancho)           = {M_u_placa:.3f} kg·cm/cm")
print(f"  t_req = √(4·M_u_placa / (F_R·F_y))        = {t_req:.4f} cm  ({t_req*10:.1f} mm)")
if t_req > 50.0:
    print("  ► ATENCIÓN: espesor muy grande — revisar hipótesis o diseño alternativo.")

print(f"\n{sep}")


CASO DE ESTUDIO 1 — FLEXOCOMPRESIÓN EXTREMA (NTC-2023, Cap. 13)

--- DATOS DE ENTRADA ---
  Perfil IR 305x74.5 : d = 30.5 cm,  b_p = 16.5 cm
  Acero A992         : F_y = 3515.0 kg/cm²
  Placa base         : N = 50.0 cm,  B = 50.0 cm
  Dado de concreto   : 60.0 × 60.0 cm
  Concreto           : f'c = 250.0 kg/cm²
  Cargas             : P_u = 12000 kg,  M_u = 1800000 kg·cm,  V_u = 4000 kg
  Anclas             : f = 20.0 cm,  n_anclas = 2
  F_R aplastamiento  : 0.65,   F_R flexión: 0.9

--- BLOQUE 1: Áreas, aplastamiento y voladizos (Sección 13.3.1 / 13.1.1.4) ---
  A1 = N × B = 2500.00 cm²
  A2 = 60.0 × 60.0 = 3600.00 cm²
  f_pu (raw) = 0.85 × f'c × √(A2/A1) = 255.000000 kg/cm²
  Límite 1.7 f'c     = 425.000000 kg/cm²
  f_pu (usado)       = 255.000000 kg/cm²
  f_pd = F_R × f_pu  = 0.65 × 255.000000 = 165.750000 kg/cm²
  m = (N - b_p)/2    = 16.7500 cm
  n = (B - d)/2      = 9.7500 cm
  l = max(m, n)      = 16.7500 cm

--- BLOQUE 2: Excentricidad (Sección 13.1.4.1) ---
  e      = M_u / P_u